# Day 7 · Transformer assignment

**Core work: 30–40 minutes, after setup.** Work alone or in pairs. This notebook runs independently of the demonstration; it does not rely on variables from another kernel. Use the [guided lab](../Transformers_what_can_they_do_.ipynb) as a reference.

**Submit:** this notebook with your completed code, your own outputs and short written answers. Required evidence is a task/model/architecture table, token-count comparison, sentiment stress test, two bounded sampled continuations, a greedy comparison and one limitation. Exact generated text need not match another learner's.

Cells marked `TODO` are deliberately incomplete. Running the starter notebook prints reminders rather than pretending the exercises are solved. Short hints follow each task. Instructor solutions are in a separate file and are not needed to begin.

## Setup - do this before the session

Use a **fresh Python 3.12 environment** with **PyTorch 2.6.0 CPU** and **Transformers 4.57.1**. These are the course pins, not a claim that they are the newest releases. Do not upgrade to v5 during the session: some older pipeline tasks, including summarization and translation, were removed from that API. A v5 migration needs different task code and a new smoke test. [Versioned pipeline documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines) · [Current translation model instructions](https://huggingface.co/Helsinki-NLP/opus-mt-fr-en)

The first model load downloads public weights and tokenizer files. The three core weight files total about **0.89 GB**, excluding packages and cache overhead. Allow **at least 3 GB of free disk space** for the core environment/cache. A recent laptop with **4 CPU cores and 8 GB RAM** is the planning assumption. Budget **5–20 minutes for first downloads and setup**, and roughly **1–5 minutes of computation after caching**; these are planning estimates, not benchmarks or guarantees. Network speed, package installation and hardware can change them substantially.

Examples use invented, non-sensitive text. The code runs model inference locally; downloading weights requires a Hub connection. Do not paste client material into a public Hub widget. No Hugging Face login or paid inference API is required for these public checkpoints.

In [1]:
# Optional one-time installation in a fresh notebook kernel, then restart it:
# %pip install "torch==2.6.0" --index-url https://download.pytorch.org/whl/cpu
# %pip install "transformers==4.57.1"
# On macOS, use this torch install line instead of the CPU-index line:
# %pip install "torch==2.6.0"

In [1]:
import os
import platform
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version

OFFLINE = False  # Change before imports; restart the kernel after changing it.
if OFFLINE:
    os.environ['HF_HUB_OFFLINE'] = '1'
os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
LOCAL_FILES_ONLY = OFFLINE or os.environ.get('HF_HUB_OFFLINE') == '1'

def require_versions(required):
    for package, expected in required.items():
        try:
            actual = version(package)
        except PackageNotFoundError as exc:
            raise RuntimeError(f'Install {package}=={expected} in this kernel, then restart it.') from exc
        print(f'{package}: {actual}')
        if actual.split('+')[0] != expected:
            raise RuntimeError(f'This lab pins {package}=={expected}; use the course environment and restart.')

require_versions({'torch': '2.6.0', 'transformers': '4.57.1'})
import torch
from transformers import AutoTokenizer, pipeline, set_seed

torch.set_num_threads(min(4, os.cpu_count() or 1))
SEED = 42
set_seed(SEED)
print('Python:', platform.python_version())
print('Device: CPU; threads:', torch.get_num_threads())
print('HF_HOME:', os.environ['HF_HOME'])
print('Local cache only:', LOCAL_FILES_ONLY)

torch: 2.6.0+cpu
transformers: 4.57.1


c:\Users\ridhan.parvendhan\Documents\GitHub\Accenture_Bootcamp\Day 7\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.10
Device: CPU; threads: 4
HF_HOME: C:\Users\ridhan.parvendhan\.cache\huggingface
Local cache only: False


In [2]:
MODEL_SPECS = {'sentiment': {'task': 'sentiment-analysis',
               'model': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english',
               'revision': '714eb0fa89d2f80546fda750413ed43d93601a13'},
 'mask': {'task': 'fill-mask',
          'model': 'distilbert/distilbert-base-uncased',
          'revision': '12040accade4e8a0f71eabdb258fecc2e7e948be'},
 'generation': {'task': 'text-generation',
                'model': 'distilbert/distilgpt2',
                'revision': '2290a62682d06624634c1f46a6ad5be0f47f38aa'}}

def load_pipeline(name, **task_options):
    spec = MODEL_SPECS[name]
    tokenizer = AutoTokenizer.from_pretrained(
        spec['model'], revision=spec['revision'],
        local_files_only=LOCAL_FILES_ONLY, trust_remote_code=False,
    )
    result = pipeline(
        task=spec['task'], model=spec['model'], tokenizer=tokenizer,
        revision=spec['revision'], framework='pt', device=-1,
        trust_remote_code=False,
        model_kwargs={'use_safetensors': True, 'local_files_only': LOCAL_FILES_ONLY},
        **task_options,
    )
    print(f"{name}: {spec['model']} @ {spec['revision'][:7]}")
    print(type(result.model).__name__, '/', type(result.tokenizer).__name__)
    return result

## Task 1 · Match the parts (5 minutes)

Complete the table. Use the three checkpoints in `MODEL_SPECS`. In the last column, describe the result's meaning rather than just its Python type.

| Task | Checkpoint | Architecture family | What the output means |
|---|---|---|---|
| Sentiment | distilbert/distilbert-base-uncased-finetuned-sst-2-english | Encoder only (DistilBERT) with a clasification head | possitive or negative with a score shows how sure the model is |
| Fill-mask | distilbert/distilbert-base-uncased | Encoder only (DistilBERT) with a masked language model head | ranked tokens that could fill the mask each with its probability out of the whole vocabulary |
| Text generation | distilbert/distilgpt2 | Decoder only (GPT-2) | continuation of the prompt, built one likely token at a time and it is not checked for facts |

**Explain in 2–3 sentences:** why does selecting a base BERT-like encoder not automatically give us a trained sentiment classifier? What must match between a tokenizer and a checkpoint?

**Success:** all three mappings are complete; your explanation distinguishes architecture, learned weights, output head and tokenizer. **Hint:** read the model card's task and training description; `pipeline` is a wrapper, not an architecture.

**Your answer:** A base enoder only has the architechture and weights pretraitned to fill the masked words and it has no trained sentiment head, so new head would have random weights until it is fine tuned on labelled sentiment data.

The tokenizer must match the checkpoint in vocabulary , token Id, special tokens and lowercasing, because the weights were learned with exactly those ID

## Task 2 · Make tokens visible (7 minutes)

Load the mask pipeline in the next cell. For `token_text`, compute a whitespace word count, token IDs **without special tokens**, and a token count. Print the token strings too. Then construct one sentence using `unmasker.tokenizer.mask_token` and request its top three completions.

Change one context word and run the mask prediction again. Record the candidate token and score that changed most noticeably. Do not guess a prediction before running it.

**Success:** you show both counts and the actual IDs/token strings; two inputs each contain exactly one valid mask; each result has three candidates. Explain why top-three scores need not sum to one.

**Hint:** `tokenizer(text, add_special_tokens=False)` returns a dictionary containing `input_ids`; `convert_ids_to_tokens` makes those IDs readable. The tokenizer exposes `mask_token`.

In [4]:
unmasker = load_pipeline('mask')
token_text = "Re-tokenization isn't always one word per token."
word_count = len(token_text.split())      # TODO
token_ids = unmasker.tokenizer(token_text,add_special_tokens=False)['input_ids']       # TODO
token_count = len(token_ids)     # TODO
masked_text = f"The students opened their {unmasker.tokenizer.mask_token} before the lesson started"     # TODO: use the tokenizer's mask marker
mask_results = unmasker(masked_text, top_k=3)    # TODO: call unmasker for three candidates

if any(value is None for value in [word_count, token_ids, token_count, masked_text, mask_results]):
    print('Complete Task 2 TODOs, then rerun this cell.')
else:
    print('Words:', word_count, 'Tokens:', token_count)
    print('Token IDs:', token_ids)
    print('Token strings:', unmasker.tokenizer.convert_ids_to_tokens(token_ids))
    print(mask_results)
    assert masked_text.count(unmasker.tokenizer.mask_token) == 1
    assert len(mask_results) == 3
    assert token_count == len(token_ids)

Device set to use cpu


mask: distilbert/distilbert-base-uncased @ 12040ac
DistilBertForMaskedLM / DistilBertTokenizerFast
Words: 7 Tokens: 13
Token IDs: [2128, 1011, 19204, 3989, 3475, 1005, 1056, 2467, 2028, 2773, 2566, 19204, 1012]
Token strings: ['re', '-', 'token', '##ization', 'isn', "'", 't', 'always', 'one', 'word', 'per', 'token', '.']
[{'score': 0.34438690543174744, 'token': 15076, 'token_str': 'mouths', 'sequence': 'the students opened their mouths before the lesson started'}, {'score': 0.1525321751832962, 'token': 4303, 'token_str': 'doors', 'sequence': 'the students opened their doors before the lesson started'}, {'score': 0.06353546679019928, 'token': 2159, 'token_str': 'eyes', 'sequence': 'the students opened their eyes before the lesson started'}]


In [7]:
changed_masked_text = f"The doctors opened their {unmasker.tokenizer.mask_token} before the lesson started"   # TODO: change one context word
changed_mask_results = unmasker(changed_masked_text, top_k = 3) # TODO: rerun the same model with top_k=3
if changed_mask_results is None:
    print('Complete the second input and compare its observed ranking.')
else:
    print(changed_masked_text, changed_mask_results)

The doctors opened their [MASK] before the lesson started [{'score': 0.5737971067428589, 'token': 15076, 'token_str': 'mouths', 'sequence': 'the doctors opened their mouths before the lesson started'}, {'score': 0.13123071193695068, 'token': 2159, 'token_str': 'eyes', 'sequence': 'the doctors opened their eyes before the lesson started'}, {'score': 0.0684816837310791, 'token': 4303, 'token_str': 'doors', 'sequence': 'the doctors opened their doors before the lesson started'}]


**Your interpretation:** The text has 7 whitespace words but 13 tokens, because the tokenizer also splits punctuation and rare words and after changing students - doctors the score of mouths changed the most from 0.344 to 0.699 doors also fell from second place 0.152 to third 0.068 and eyes moved up so the context word changes the ranking and the probablities added cant give 1 since the rest belongs to tokens that are not shown

## Task 3 · Stress-test a classifier (7 minutes)

Before running the model, label the four examples yourself as positive, negative, neutral or ambiguous. Then call the sentiment pipeline on the entire list, with input truncation enabled and a 128-token input cap. Compare your assessment with the model's available labels.

**Success:** four observed predictions with scores; one disagreement or ambiguous case discussed; one sentence explaining why a high score does not validate use on client tickets. If you agree with all four outputs, add an invented neutral or mixed-sentiment example and discuss the forced binary choice.

**Hint:** this checkpoint has no neutral output class. Inspect `sentiment.model.config.id2label`. A tiny hand-picked sample illustrates a failure mode; it is not a benchmark.

In [8]:
sentiment = load_pipeline('sentiment')
examples = [
    'The explanation was clear and useful.',
    'The application crashes every time I open it.',
    "The meeting starts at two o'clock.",
    'I loved the tutor, but the exercises were frustrating.',
]
my_labels = ['positive','negative','neutral','ambiguous']              # TODO: four judgments before model inference
student_predictions = sentiment(examples, truncation=True, max_length=128) # TODO: one pipeline call on examples
print('Available model labels:', sentiment.model.config.id2label)
if student_predictions is None or len(my_labels) != len(examples):
    print('Complete your judgments and the pipeline call.')
else:
    assert len(student_predictions) == len(examples)
    for text, human, model_result in zip(examples, my_labels, student_predictions):
        print({'text': text, 'my_judgment': human, 'model': model_result})

c:\Users\ridhan.parvendhan\Documents\GitHub\Accenture_Bootcamp\Day 7\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ridhan.parvendhan\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled 

sentiment: distilbert/distilbert-base-uncased-finetuned-sst-2-english @ 714eb0f
DistilBertForSequenceClassification / DistilBertTokenizerFast
Available model labels: {0: 'NEGATIVE', 1: 'POSITIVE'}
{'text': 'The explanation was clear and useful.', 'my_judgment': 'positive', 'model': {'label': 'POSITIVE', 'score': 0.9997910857200623}}
{'text': 'The application crashes every time I open it.', 'my_judgment': 'negative', 'model': {'label': 'NEGATIVE', 'score': 0.9997283816337585}}
{'text': "The meeting starts at two o'clock.", 'my_judgment': 'neutral', 'model': {'label': 'POSITIVE', 'score': 0.9768263697624207}}
{'text': 'I loved the tutor, but the exercises were frustrating.', 'my_judgment': 'ambiguous', 'model': {'label': 'NEGATIVE', 'score': 0.9912058711051941}}


**Your interpretation:** Example 1 and 2 matched the failed case is example 3 I lablled it neutral but the model said positive with 0.977 becuase it has no neutral class and only positive and negative have to pick from either of this two and example 4 is kind of a mixed but the model gave negative with 0.991 probablity and it seems ignored the negative part


## Task 4 · Bound generation and compare decoding (10 minutes)

Load the generation model and run the provided reporting helper. Write a call that generates **two sampled continuations with at most 15 new tokens each**. Use a seed of 42, one beam and deliberate sampling parameters. Request token IDs with `return_tensors=True`; pass the tokenizer's EOS ID as `pad_token_id` for this model.

Then generate **one greedy continuation** with the same prompt and token cap. Print all three continuations with both token counts and whitespace word counts.

**Success:** two sampled results, one greedy result, and each count at most 15. Your explanation says “maximum new tokens,” allows early stopping and does not require exactly 15 words, two complete sentences or different sampled texts.

**Hint:** inspect `max_new_tokens`, `num_return_sequences` and `do_sample`. Greedy decoding with one beam uses one return sequence. Tune `temperature` and `top_p` for sampling. For the greedy call, use neutral `temperature=1.0` to reset this v4 pipeline's inherited 0.7 default and avoid an ignored-parameter warning; it has no sampling effect when `do_sample=False`.

In [9]:
def describe_generations(results, prompt, tokenizer):
    """Count original generated IDs, rather than re-tokenizing decoded text."""
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    rows = []
    for result in results:
        all_ids = result['generated_token_ids']
        assert all_ids[:len(prompt_ids)] == prompt_ids
        added_ids = all_ids[len(prompt_ids):]
        # DistilGPT2 uses EOS as padding here; ignore padding after the first EOS.
        if tokenizer.eos_token_id in added_ids:
            added_ids = added_ids[:added_ids.index(tokenizer.eos_token_id) + 1]
        continuation = tokenizer.decode(added_ids, skip_special_tokens=True)
        rows.append({
            'continuation': continuation,
            'new_tokens_including_eos': len(added_ids),
            'whitespace_words': len(continuation.split()),
        })
    return rows

In [11]:
generator = load_pipeline('generation')
prompt = 'To prepare a useful training workshop, we should'
set_seed(SEED)
sampled_results = generator(prompt,max_new_tokens=15,num_return_sequences = 2,
                            do_sample=True, temperature=0.8,top_p=0.9,num_beams = 1,return_tensors=True,
                            pad_token_id=generator.tokenizer.eos_token_id,)  # TODO: two sampled continuations, return token IDs
greedy_results = generator(prompt,max_new_tokens=15,num_return_sequences = 1,
                            do_sample=False, temperature=1.0,num_beams = 1,return_tensors=True,
                            pad_token_id=generator.tokenizer.eos_token_id,)   # TODO: one greedy continuation, same token cap

Device set to use cpu


generation: distilbert/distilgpt2 @ 2290a62
GPT2LMHeadModel / GPT2TokenizerFast


In [12]:
if sampled_results is None or greedy_results is None:
    print('Complete both generation calls, then rerun this check.')
else:
    sampled_rows = describe_generations(sampled_results, prompt, generator.tokenizer)
    greedy_rows = describe_generations(greedy_results, prompt, generator.tokenizer)
    assert len(sampled_rows) == 2
    assert len(greedy_rows) == 1
    assert all(1 <= row['new_tokens_including_eos'] <= 15 for row in sampled_rows + greedy_rows)
    print('Sampled:', sampled_rows)
    print('Greedy:', greedy_rows)

Sampled: [{'continuation': ' try to introduce some basic knowledge about the different methods used for training.\n', 'new_tokens_including_eos': 15, 'whitespace_words': 13}, {'continuation': ' be able to use this tool to help us train in a more efficient manner', 'new_tokens_including_eos': 15, 'whitespace_words': 15}]
Greedy: [{'continuation': ' be able to use the following techniques to prepare a useful training workshop.\n', 'new_tokens_including_eos': 15, 'whitespace_words': 13}]


**Your interpretation:** All three continuations used the maximum of 15 new tokens and the sampled ones had 13 and 15 whitespace words and were different from each other and The greedy one had 13 words and started like the second sample

(a) The limit is maximum new tokens not words a token can be a word, a part of a word, punctuation or a line break so the word count is usually lower and the model may also stop early

(b) Greedy always takes the most likely next token therefore it gives the same text every time and sampling picks randomly from the likely tokens so results can differ but they are not guaranteed to differ

(c) Nothing checks the text is true since both are choose tokens that are likely after the prompt

For exactly 15 words the application must count in code after generation: generate extra tokens, split on whitespace, keep the first 15 words, and retry if there are fewer.


## Task 5 · Make a model decision (5 minutes)

A team wants to use one of these checkpoints on a new task. Choose **one** scenario and write 4–6 sentences:

- Route short English customer-support messages to three departments.
- Suggest a continuation for public training copy.
- Extract named organizations from public event announcements.

Name a suitable task and model family, identify a mismatch with at least one of the core checkpoints, give one test case, and state one evaluation measure or review criterion. Also identify a case the system should hand to a human. You do not need to download another model.

**Success:** your recommendation follows the task and evidence; it does not use a sentiment score as routing accuracy or fluency as factual correctness.

**Your answer:** I choose routing short support messages to three departments, the suitatble task is clasificaiton with threeclasses, using an encoder model that is finetuned on messages labelled by department, the sentiment checkpoint is a mismatch because it only outputs positive or negarive and angry billing message and an angry technical message would get the same label A test case is "I was charged twice this month" which should go to billing then I would measure accuracy and F1 for each department on a hand labelled set to test and the messages with low confidence or fit two departments should go to the human for review.

## Optional investigation - choose one (10–15 minutes)

These preserve the original exploration topics. A model-card review is enough for the first pass; download only after checking size and language support. If you run a model, use an explicit model ID and record its commit revision, task, tokenizer, library version and actual output. If you do not run it, state **not executed** and explain the planned test.

| Option | Action | Evidence to submit |
|---|---|---|
| Another language | Find a text-generation model whose card supports a language you know. | Model URL, language, architecture, task, weight size, limitation and one bounded test prompt. |
| Cased BERT | Inspect [google-bert/bert-base-cased](https://huggingface.co/google-bert/bert-base-cased). Identify its mask marker and contrast cased/uncased tokenization. | The marker and source; if executed, the top three observed candidates for one input. Do not copy a prediction from a widget. |
| POS tagging | Inspect [vblagoje/bert-english-uncased-finetuned-pos](https://huggingface.co/vblagoje/bert-english-uncased-finetuned-pos) or another English POS checkpoint. | Task, at least three tag meanings, and why those labels differ from NER. Note any missing license/training information in the card. |
| Translation | Choose one directed language pair and inspect a suitable model card. | Source/target language, tokenizer requirements, model URL and a plan to check numbers, names and meaning. |
| Original advanced demos | Enable one optional section in the guided lab: zero-shot, NER, QA or summary. | Your observed result and one limitation demonstrated by changing the input. |

**Your investigation and execution status:** 
Cased BERT
- Model: google-bert/bert-base-cased (https://huggingface.co/google-bert/bert-base-cased), revision `cd5ef92a9fb2f889e972770a36d4ed042daf221e`, task fill-mask, tokenizer `BertTokenizerFast`, Transformers 4.57.1.
- Mask marker: `[MASK]`, read from `tokenizer.mask_token` 
- Cased versus uncased: the cased tokenizer keeps capital letters, so "Paris" is one token but "paris" is split into `par`, `##is`. The uncased tokenizer lowercases first, so both become the same token `paris`.
- Top three observed candidates for "The capital of France is [MASK].": `Paris` (0.4447), `Lyon` (0.0940), `Toulouse` (0.0823).
- Limitation: the model gives likely word that not checked facts. In my execution the first answer is correct here, but Lyon and Toulouse also get a score, so the output still needs checking.


In [13]:
import transformers
CASED_ID = 'google-bert/bert-base-cased'
CASED_REV = 'cd5ef92a9fb2f889e972770a36d4ed042daf221e'
cased_tok = AutoTokenizer.from_pretrained(CASED_ID, revision=CASED_REV,
                                          local_files_only=LOCAL_FILES_ONLY, trust_remote_code=False)
cased_unmasker = pipeline(task='fill-mask', model=CASED_ID, tokenizer=cased_tok, revision=CASED_REV,
                          framework='pt', device=-1, trust_remote_code=False,
                          model_kwargs={'use_safetensors': True, 'local_files_only': LOCAL_FILES_ONLY})
print('Model:', CASED_ID, '@', CASED_REV[:7], '| task: fill-mask | transformers', transformers.__version__)
print(type(cased_unmasker.model).__name__, '/', type(cased_tok).__name__)
print('Mask marker:', cased_tok.mask_token)

case_text = 'Paris and paris are not the same for a Cased model.'
print('Cased  :', cased_tok.tokenize(case_text))
print('Uncased:', unmasker.tokenizer.tokenize(case_text))

cased_masked_text = f'The capital of France is {cased_tok.mask_token}.'
cased_results = cased_unmasker(cased_masked_text, top_k=3)
assert cased_masked_text.count(cased_tok.mask_token) == 1 and len(cased_results) == 3
for r in cased_results:
    print(round(r['score'], 4), r['token_str'], '|', r['sequence'])



c:\Users\ridhan.parvendhan\Documents\GitHub\Accenture_Bootcamp\Day 7\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ridhan.parvendhan\.cache\huggingface\hub\models--google-bert--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet'

Model: google-bert/bert-base-cased @ cd5ef92 | task: fill-mask | transformers 4.57.1
BertForMaskedLM / BertTokenizerFast
Mask marker: [MASK]
Cased  : ['Paris', 'and', 'par', '##is', 'are', 'not', 'the', 'same', 'for', 'a', 'Case', '##d', 'model', '.']
Uncased: ['paris', 'and', 'paris', 'are', 'not', 'the', 'same', 'for', 'a', 'case', '##d', 'model', '.']
0.4447 Paris | The capital of France is Paris.
0.094 Lyon | The capital of France is Lyon.
0.0823 Toulouse | The capital of France is Toulouse.


## Submission check

- Replace every required `TODO` with your own code or interpretation.
- Restart and run the notebook in order using the pinned kernel. Keep the printed versions.
- Keep your observed outputs; do not invent a plausible result for an unexecuted cell.
- Explain one failure or ambiguity even when the code runs successfully.
- Mark the optional investigation as executed or not executed.

**Assessment:** task/model mapping (2 points), token and mask evidence (2), sentiment interpretation (2), bounded generation and decoding comparison (3), model decision and limitation (1). Total: **10 points**. Optional work is for discussion, not required to achieve 10/10.

## Sources and attribution

The original notebook follows the Hugging Face course chapter [Transformers, what can they do?](https://huggingface.co/learn/llm-course/chapter1/3). This revision uses new classroom examples and adds setup, model selection, bounded generation, exercises and interpretation. This is **source attribution**, not a license grant; the course source and each model retain their own applicable licenses. Model weights are downloaded separately and are not distributed in this notebook.

Primary references checked for this revision:

- [Transformers 4.57.1 pipeline API](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines), including tasks, NER aggregation and output formats.
- [Transformers 4.57.1 generation parameters](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/text_generation), including sampling and token limits.
- [Installation, caching and offline mode](https://huggingface.co/docs/transformers/v4.57.1/en/installation).
- [Sentiment model card](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) · [masked-language model card](https://huggingface.co/distilbert/distilbert-base-uncased) · [DistilGPT2 model card](https://huggingface.co/distilbert/distilgpt2).
- Weight-size evidence: [sentiment files](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main), [mask-model files](https://huggingface.co/distilbert/distilbert-base-uncased/tree/main), [generation-model files](https://huggingface.co/distilbert/distilgpt2/tree/main). The figures are for one safetensors weight file, not the entire repository or runtime memory.

Model revision hashes in the code freeze the reviewed files. A fixed seed improves repeatability in a fixed environment; it does not guarantee bit-identical text across different hardware, library versions or decoding implementations. Inspect warnings; this notebook does not suppress them globally.